In [ ]:
from app.src.core_etl import get_html_content_metadata, refine_metadata_with_llm, run_pipeline
import asyncio

# `extract_html_data` ahora incluye un fallback que usa Playwright si es necesario

In [ ]:
TEST_URL = "https://www.apa.org/topics/psychotherapy/psicoterapia"

# Ejecutamos la extracción principal (usa requests/cloudscraper y Playwright como fallback)
"""result = await get_html_content_metadata(TEST_URL)

# Mostrar un resumen rápido
print('METADATA_SEO:', result.get('metadata_seo'))
print('TEXTO_LIMPIO length:', len(result.get('texto_limpio', '')))
print('TEXTO_LIMPIO (primeros 1000 chars):')
print(result.get('content', ''))"""

"result = await get_html_content_metadata(TEST_URL)\n\n# Mostrar un resumen rápido\nprint('METADATA_SEO:', result.get('metadata_seo'))\nprint('TEXTO_LIMPIO length:', len(result.get('texto_limpio', '')))\nprint('TEXTO_LIMPIO (primeros 1000 chars):')\nprint(result.get('content', ''))"

In [ ]:
await run_pipeline(TEST_URL)

[*] Descargando HTML de: https://retinia.mx/complicaciones-visuales/
[*] Contenido sin cambios, saltando.

=== PRUEBA DE PIPELINE EXITOSA ===


In [4]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
import os
import dotenv
dotenv.load_dotenv()
# Configurar el cliente de búsqueda
endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
key = os.getenv("AZURE_SEARCH_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX_NAME")

client = SearchClient(endpoint, index_name, AzureKeyCredential(key))

# Realizar una búsqueda de conteo
results = client.search(search_text="*", include_total_count=True)
print(f"Total de documentos en el índice: {results.get_count()}")

# Ver un ejemplo de cómo guardó Azure tu documento
for result in client.search(search_text="calabaza"):
    print(f"Encontrado: {result['id']}")

Total de documentos en el índice: 17
Encontrado: e02998dc74e3bd2c48350b1185cf1bf5
Encontrado: 4e2ba7741f897f53698af23479a19320
Encontrado: 87a53a65d965e83b18109378b9aa335f


In [12]:
result

{'@search.score': 0.3812864,
 'id': '87a53a65d965e83b18109378b9aa335f',
 'content': 'Refrigere las sobras dentro de las siguientes 2 horas. Notas:\n\nAgréguele granola o trocitos de cereal para un licuado extra crujiente. Puede congelar la calabaza tipo "pumpkin" enlatada que sobre para usarla después en platos principales, sopas, chile o alimentos cocidos. Encuentre más recetas deliciosas y saludables del FoodHero.org\n\n \n\n\t\nInformación nutricional\n1 porción por envase\nTamaño por ración\n1 taza (241g)\nCantidad por ración\nCalorías\n200\n% Valor Diario*\nGrasa Total 2.5g\n3%\nGrasa Saturada 1.5g\n8%\nGrasa Trans 0g\nColesterol 10mg\n3%\nSodio 120mg\n5%\nCarbohidrato Total 38g\n14%\nFibra Dietética 2g\n7%\nAzúcares Totales 34g\nIncluye 20g azúcares añadidos\n40%\nProteínas 9g\nVitamina A 497mcg\n55%\nVitamina C 4mg\n4%\nVitamina D 0mcg\n0%\nCalcio 321mg\n25%\nHierro 1mg\n6%\nPotasio 514mg\n10%\n* El % Valor Diario (VD) le indica cuánto un nutriente en una porción de alimentos co

In [4]:
import asyncio
from app.src.core_etl import run_pipeline

async def batch_processor(urls):
    semaphore = asyncio.Semaphore(5)
    
    async def process_with_limit(url):
        async with semaphore:
            print(f"[*] Iniciando procesamiento de: {url}")
            await run_pipeline(url)

    tasks = [process_with_limit(url) for url in urls]
    await asyncio.gather(*tasks)

urls_de_prueba = [
        "https://medlineplus.gov/spanish/recetas/licuado-de-calabaza/",
        "https://retinia.mx/agudeza-visual-av/",
        "https://www.cdc.gov/spanish/niosh/topics/tiro.html",
        "http://familiasysexualidades.inmujeres.gob.mx/pdf/10.pdf",
        "https://medlineplus.gov/spanish/druginfo/meds/a622022-es.html"
        
    ]
await batch_processor(urls=urls_de_prueba)

[*] Iniciando procesamiento de: https://medlineplus.gov/spanish/recetas/licuado-de-calabaza/
[*] Descargando HTML de: https://medlineplus.gov/spanish/recetas/licuado-de-calabaza/
[*] Iniciando procesamiento de: https://retinia.mx/agudeza-visual-av/
[*] Descargando HTML de: https://retinia.mx/agudeza-visual-av/
[*] Contenido sin cambios, saltando.

=== PRUEBA DE PIPELINE EXITOSA ===
[*] Iniciando procesamiento de: https://www.cdc.gov/spanish/niosh/topics/tiro.html
[*] Descargando HTML de: https://www.cdc.gov/spanish/niosh/topics/tiro.html
[-] Error al descargar https://www.cdc.gov/spanish/niosh/topics/tiro.html: 404 Client Error: Not Found for url: https://www.cdc.gov/spanish/niosh/topics/tiro.html
[*] Contenido sin cambios, saltando.

=== PRUEBA DE PIPELINE EXITOSA ===
[*] Iniciando procesamiento de: http://familiasysexualidades.inmujeres.gob.mx/pdf/10.pdf
[*] Descargando HTML de: http://familiasysexualidades.inmujeres.gob.mx/pdf/10.pdf
[-] Error al descargar http://familiasysexualidad